# 07 · Trunk Fit & Dijkstra Branch Graph

**Goal:** turn the filtered point cloud into a structured branch graph.

From scratch (`src/trunk.py`, `src/branch_graph.py`):

1. **`ransac_trunk_axis`** — RANSAC 3D line fit with a near-vertical prior.
2. **`build_knn_graph`** — adjacency list with Euclidean weights.
3. **`dijkstra_shortest_path_tree`** — heap-based Dijkstra from the trunk
   root.
4. **`prune_spurs`** — drop tiny noise branches.
5. **`merge_collinear_segments`** — collapse straight runs (3D analogue of
   Hough-line merging from Lec. 3).

Why a shortest-path tree? Real trees branch and never rejoin, so the
geodesic ancestry of any point IS its branch lineage. (Livny et al., 2010.)

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import sfm, trunk, branch_graph, viz

TREE_ID = "tree_oak_01"
MASK_VARIANT = "sam"
FILTERED_PLY = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_filtered_{MASK_VARIANT}.ply"


if not FILTERED_PLY.exists():
    raise FileNotFoundError(
        f"Filtered cloud PLY not found at {FILTERED_PLY}.\n"
        "Run notebook 06 first."
    )


points, _ = sfm.load_ply(FILTERED_PLY)
print(f"Loaded {len(points)} filtered points.")


## 1. Fit the trunk axis

In [ ]:
fit = trunk.ransac_trunk_axis(
    points,
    num_iter=1000,
    inlier_thresh=0.05,
    vertical_prior=True,
    vertical_axis=2,         # adjust if COLMAP's up axis ended up Y instead of Z
)
n_inl = int(fit.inlier_mask.sum())
print(f"Trunk fit: {n_inl}/{len(points)} inliers, mean perp dist = {fit.inlier_distance_mean*100:.1f} cm")
print(f"Trunk root: {fit.root}, direction (unit): {fit.direction}")
print(f"Trunk height: {trunk.trunk_height(fit, points):.2f} m")

## 2. Extract the branch graph

In [ ]:
graph = branch_graph.extract_branch_graph(
    points,
    trunk_root_xyz=fit.root,
    k=8,
    max_edge_length=0.30,
    min_spur_length=0.10,
    collinear_tolerance_deg=8.0,
)
print(f"Skeleton: {len(graph.nodes)} nodes, {len(graph.edges)} edges")

## 3. Visualise — trunk + skeleton overlaid on the filtered cloud

In [ ]:
fig = viz.plot_branch_graph(
    graph.nodes, graph.edges,
    trunk_axis=(fit.root, fit.direction),
    title=f"{TREE_ID} — branch graph ({len(graph.edges)} edges)",
)
viz.save_fig(fig, f"07_{TREE_ID}_branch_graph.png")
plt.show()

## 4. Persist the graph for evaluation

In [ ]:
out = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_graph.npz"
np.savez(
    out,
    nodes=graph.nodes,
    edges=np.array(graph.edges, dtype=np.int64),
    trunk_root=fit.root, trunk_direction=fit.direction,
    parents=np.array(list(graph.parent_of.items()), dtype=np.int64),
)
print(f"Saved → {out}")